In [2]:
import sys
import os

# --- Bước 1: Thêm thư mục gốc của dự án vào Python Path ---

# Lấy đường dẫn thư mục hiện tại của notebook (.../lesson-03/notebook)
current_dir = os.getcwd()

# Đi lùi 2 cấp để đến thư mục gốc của dự án (.../DS201-DL-PRACTICALLESSON)
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# Thêm thư mục gốc vào sys.path nếu nó chưa có ở đó
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Đã thêm vào sys.path: {project_root}")


Đã thêm vào sys.path: c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\lesson-03


In [4]:
from src.models.lstm import lstm
from src.utils import Vocab

from src.train import train

import torch
import torch.nn as nn
import pandas as pd
from functools import partial
from torch.utils.data import DataLoader

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Session 00

## Session 01

### 1.1. Requirements

> Xây dựng mạng LSTM gồm 5 lớp với hidden size là 256 cho bài toán phân loại văn bản. Huấn luyện mô hình này trên bộ dữ liệu UIT-VSFC (Vietnamese Student Feedback Corpus) sử dụng Adam làm phương thức tối ưu tham số và đánh giá độ hiệu quả của mô hình sử dụng độ đo F1.

### 1.2. Configuration

#### 1.2.1. Vocabulary

In [7]:
vocab_path = r'..\dataset\uit_vsvc'
vocab = Vocab(
    vocab_path
)

In [8]:
PAD_IDX = vocab.w2i['<PAD>']

#### 1.2.2. Dataset

In [9]:
from src.utils import VsvcDataset, collate_fn

In [10]:
try:
    _train_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-train.json')
    _dev_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-dev.json')
    _test_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-test.json')
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file train/dev tại '{vocab_path}'")

train_dataset = VsvcDataset(dataframe=_train_vsvc, vocab=vocab)
dev_dataset = VsvcDataset(dataframe=_dev_vsvc, vocab=vocab)
test_dataset = VsvcDataset(dataframe=_test_vsvc, vocab=vocab)

In [11]:
collate_fn_with_padding = partial(collate_fn, pad_idx=PAD_IDX)

# Tạo loader
vsvc_train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn_with_padding
)

vsvc_dev_loader = DataLoader(
    dataset=dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding
)

vsvc_test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding 
)

### 1.3. Training

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
lstm_instance = lstm(
    vocab_size=vocab.vocab_size,
    embedding_dim=256,
    hidden_size=256,
    output_size=vocab.n_labels
).to(device)

print(lstm_instance)

lstm(
  (embedding): Embedding(2879, 256, padding_idx=0)
  (lstm): LSTM(256, 256, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)


In [14]:
optimizer = torch.optim.Adam(lstm_instance.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 50
N_LABELS = vocab.n_labels 
MODEL_PATH   = r"..\checkpoints\lstm\lstm_best_model.pth"
HISTORY_PATH = r"..\checkpoints\lstm\training_history.json"

In [15]:
best_model, history_data = train(
    model=lstm_instance,
    train_loader=vsvc_train_loader,
    val_loader=vsvc_dev_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=NUM_EPOCHS,
    n_labels=N_LABELS,
    model_save_path=MODEL_PATH,
    history_save_path=HISTORY_PATH,
    patience=10
)

--- Bắt đầu training ---
Lưu model tốt nhất tại: ..\checkpoints\lstm\lstm_best_model.pth
Lưu lịch sử training tại: ..\checkpoints\lstm\training_history.json


Epoch 1/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 93.31it/s] 


Epoch 1: Train Loss: 0.8551 | Val Loss: 0.8383 | Val F1: 0.7271
🎉 New best F1: 0.7271. Model saved to ..\checkpoints\lstm\lstm_best_model.pth


Epoch 2/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 141.85it/s]


Epoch 2: Train Loss: 0.7456 | Val Loss: 0.6764 | Val F1: 0.7720
🎉 New best F1: 0.7720. Model saved to ..\checkpoints\lstm\lstm_best_model.pth


...

Epoch 20: Train Loss: 0.0639 | Val Loss: 0.5679 | Val F1: 0.8572
No improvement. Patience: 8/10


Epoch 21/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 151.14it/s]


Epoch 21: Train Loss: 0.0597 | Val Loss: 0.5990 | Val F1: 0.8534
No improvement. Patience: 9/10


Epoch 22/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 153.28it/s]

Epoch 22: Train Loss: 0.0513 | Val Loss: 0.6164 | Val F1: 0.8617
No improvement. Patience: 10/10
Early stopping triggered after 22 epochs.

--- Training finished ---
Best Validation F1-score: 0.8711
Training history successfully saved to ..\checkpoints\lstm\training_history.json


### 1.4. Evaluation

In [16]:
from src.evaluate import evaluate

In [17]:
label_names = [vocab.i2l[i] for i in range(vocab.n_labels)]
print(f"Các nhãn (theo thứ tự): {label_names}")

Các nhãn (theo thứ tự): ['training_program', 'lecturer', 'others', 'facility']


In [18]:
test_results = evaluate(
    model=best_model,
    test_loader=vsvc_test_loader,
    criterion=criterion,
    device=device,
    n_labels=N_LABELS,
    label_names=label_names
)

--- Bắt đầu đánh giá trên tập Test ---


Evaluating:   0%|          | 0/99 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 99/99 [00:00<00:00, 116.09it/s]


--- 🏁 Kết quả Đánh giá trên tập Test ---
Thời gian đánh giá: 0.86 giây
Test Loss: 	0.4019
Test Accuracy: 	86.86%
Test F1-Score (Macro): 	0.7482

📊 Báo cáo chi tiết (Classification Report):
                  precision    recall  f1-score   support

training_program       0.73      0.74      0.73       572
        lecturer       0.92      0.93      0.93      2290
          others       0.48      0.40      0.44       159
        facility       0.87      0.91      0.89       145

        accuracy                           0.87      3166
       macro avg       0.75      0.75      0.75      3166
    weighted avg       0.87      0.87      0.87      3166



## Session 02

### 2.1. Requirements

Xây dựng mạng GRU gồm 5 lớp với hidden size là 256 cho bài toán phân loại văn bản. Huấn luyện mô hình này trên bộ dữ liệu UIT-VSFC (Vietnamese Student Feedback Corpus) sử dụng Adam làm phương thức tối ưu tham số và đánh giá độ hiệu quả của mô hình sử dụng độ đo F1.

### 2.2. Configuration

#### 2.2.1. Vocabulary

In [19]:
vocab_path = r'..\dataset\uit_vsvc'
vocab = Vocab(
    vocab_path
)
PAD_IDX = vocab.w2i['<PAD>']

#### 2.2.2. Dataset

In [20]:
from src.utils import VsvcDataset, collate_fn

In [21]:
try:
    _train_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-train.json')
    _dev_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-dev.json')
    _test_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-test.json')
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file train/dev tại '{vocab_path}'")

train_dataset = VsvcDataset(dataframe=_train_vsvc, vocab=vocab)
dev_dataset = VsvcDataset(dataframe=_dev_vsvc, vocab=vocab)
test_dataset = VsvcDataset(dataframe=_test_vsvc, vocab=vocab)

In [22]:
collate_fn_with_padding = partial(collate_fn, pad_idx=PAD_IDX)

# Tạo loader
vsvc_train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn_with_padding
)

vsvc_dev_loader = DataLoader(
    dataset=dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding
)

vsvc_test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding 
)

### 2.3. Training

In [23]:
from src.models.gru import gru

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [41]:
optimizer = torch.optim.Adam(lstm_instance.parameters(), lr=0.5)
criterion = nn.CrossEntropyLoss()

N_LABELS = vocab.n_labels 
GRU_NUM_EPOCHS = 50
GRU_MODEL_PATH = r'../checkpoints/gru/gru_best_model.pth'
GRU_HISTORY_PATH = r'../checkpoints/gru/training_history.json'

In [42]:
gru_instance = gru(
    vocab_size = vocab.vocab_size, 
    embedding_dim = 256, 
    hidden_size = 256, 
    num_layers = 5,
    num_classes = vocab.n_labels,
    dropout=0.5
).to(device)

print(gru_instance)

gru(
  (embedding): Embedding(2879, 256, padding_idx=0)
  (gru): GRU(256, 256, num_layers=5, batch_first=True, dropout=0.5)
  (fc): Linear(in_features=256, out_features=4, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [43]:
from src.train import train

In [44]:
gru_best_model, gru_history_data = train(
    model=gru_instance,                 #
    train_loader=vsvc_train_loader,     
    val_loader=vsvc_dev_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=GRU_NUM_EPOCHS,              #
    n_labels=N_LABELS,                      #
    model_save_path=GRU_MODEL_PATH,         #
    history_save_path=GRU_HISTORY_PATH,     #
    patience=10
)

--- Bắt đầu training ---
Lưu model tốt nhất tại: ../checkpoints/gru/gru_best_model.pth
Lưu lịch sử training tại: ../checkpoints/gru/training_history.json


Epoch 1/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 81.31it/s]


Epoch 1: Train Loss: 1.4187 | Val Loss: 1.4187 | Val F1: 0.0442
🎉 New best F1: 0.0442. Model saved to ../checkpoints/gru/gru_best_model.pth


Epoch 2/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 83.22it/s]


Epoch 2: Train Loss: 1.4188 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 1/10


...

Epoch 9: Train Loss: 1.4182 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 8/10


Epoch 10/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 72.24it/s]


Epoch 10: Train Loss: 1.4186 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 9/10


Epoch 11/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 84.55it/s]

Epoch 11: Train Loss: 1.4186 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 10/10
Early stopping triggered after 11 epochs.

--- Training finished ---
Best Validation F1-score: 0.0442
Training history successfully saved to ../checkpoints/gru/training_history.json


## Session 03

### 3.1. Requirements

Xây dựng kiến trúc Encoder-Decoder trong đó BiEncoder gồm 5 lớp LSTM và Decoder gồm 5 lớp LSTM với hidden size là 256 cho bài toán nhận diện thực thể (Name Entity Recognition). Huấn luyện mô hình trên bộ dữ liệu PhoNER và đánh giá độ hiệu quả của mô hình sử dụng độ đo F1.

### 3.2. Configuration

In [5]:
from src.models.bilstm import Encoder, Decoder, Seq2SeqNER
INPUT_DIM = 10000
OUTPUT_DIM = 15
ENC_EMB_DIM = 100
DEC_EMB_DIM = 100
HIDDEN_DIM = 256
N_LAYERS = 5
DROPOUT = 0.5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)

model = Seq2SeqNER(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)
print(f'The model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

The model has 10,780,971 trainable parameters


In [7]:
from seqeval.metrics import f1_score, classification_report
import numpy as np
from torch import optim

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0) # Giả sử 0 là PAD_TOKEN_ID

def train_epoch(model, iterator, clip):
    model.train()
    epoch_loss = 0
    
    for i, batch in enumerate(iterator):
        src = batch.text # [batch, seq_len]
        trg = batch.tags # [batch, seq_len]
        
        optimizer.zero_grad()
        output = model(src, trg)
        
        # Reshape để tính loss
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim) # Bỏ qua <SOS>
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

def evaluate_f1(model, iterator, idx2tag):
    model.eval()
    pred_tags = []
    true_tags = []
    
    with torch.no_grad():
        for batch in iterator:
            src = batch.text
            trg = batch.tags
            
            output = model(src, trg, teacher_forcing_ratio=0) # Tắt teacher forcing
            output = output.argmax(dim=-1) # [batch, seq_len]
            
            # Convert ID về Tag chuỗi (B-PER, O...) để dùng seqeval
            for i in range(output.shape[0]):
                # Lấy tags từ index 1 (bỏ SOS) và cắt padding
                p_sent = [idx2tag[idx.item()] for idx in output[i, 1:] if idx.item() in idx2tag]
                t_sent = [idx2tag[idx.item()] for idx in trg[i, 1:] if idx.item() in idx2tag]
                
                # Đảm bảo độ dài khớp nhau (do cắt padding logic có thể khác)
                min_len = min(len(p_sent), len(t_sent))
                pred_tags.append(p_sent[:min_len])
                true_tags.append(t_sent[:min_len])

    # Tính F1 score bằng seqeval
    f1 = f1_score(true_tags, pred_tags)
    print(classification_report(true_tags, pred_tags))
    return f1

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
from seqeval.metrics import f1_score, classification_report
import random
import time

# ==========================================
# 1. DATA PREPROCESSING & VOCABULARY
# ==========================================

class Vocabulary:
    def __init__(self, token_to_idx=None):
        if token_to_idx is None:
            self.token_to_idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
            self.idx_to_token = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        else:
            self.token_to_idx = token_to_idx
            self.idx_to_token = {v: k for k, v in token_to_idx.items()}

    def add_token(self, token):
        if token not in self.token_to_idx:
            idx = len(self.token_to_idx)
            self.token_to_idx[token] = idx
            self.idx_to_token[idx] = token

    def __len__(self):
        return len(self.token_to_idx)

    def encode(self, text_seq):
        return [self.token_to_idx.get(token, self.token_to_idx["<UNK>"]) for token in text_seq]

    def decode(self, idx_seq):
        return [self.idx_to_token.get(idx, "<UNK>") for idx in idx_seq]

def read_phoner_data(file_path):
    """
    Giả định định dạng file:
    Word1 Tag1
    Word2 Tag2
    ... (dòng trống ngăn cách câu)
    """
    sentences = []
    tags = []
    current_sent = []
    current_tags = []

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    if current_sent:
                        sentences.append(current_sent)
                        tags.append(current_tags)
                        current_sent = []
                        current_tags = []
                else:
                    parts = line.split() # Hoặc split('\t') tùy file
                    if len(parts) >= 2:
                        word = parts[0]
                        tag = parts[-1]
                        current_sent.append(word)
                        current_tags.append(tag)
            # Thêm câu cuối nếu file không kết thúc bằng dòng trống
            if current_sent:
                sentences.append(current_sent)
                tags.append(current_tags)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {file_path}. Đang tạo dữ liệu giả lập để test...")
        return get_dummy_data()

    return sentences, tags

def get_dummy_data():
    sents = [["Tôi", "là", "Long", "sinh", "viên", "Đại_học", "Bách_Khoa"], 
             ["Hà_Nội", "là", "thủ_đô", "của", "Việt_Nam"]]
    tags = [["O", "O", "B-PER", "O", "O", "B-ORG", "I-ORG"], 
            ["B-LOC", "O", "O", "O", "B-LOC"]]
    return sents * 50, tags * 50 # Nhân bản lên để train thử

def build_vocab(sentences, tags):
    word_vocab = Vocabulary()
    tag_vocab = Vocabulary(token_to_idx={"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3})
    # Tag không cần UNK vì ta biết hết các tag, nhưng cần SOS/EOS cho Decoder

    for sent in sentences:
        for word in sent:
            word_vocab.add_token(word)
    
    for tag_seq in tags:
        for tag in tag_seq:
            tag_vocab.add_token(tag)
            
    return word_vocab, tag_vocab

# ==========================================
# 2. DATASET & DATALOADER
# ==========================================

class NERDataset(Dataset):
    def __init__(self, sentences, tags, word_vocab, tag_vocab):
        self.sentences = sentences
        self.tags = tags
        self.word_vocab = word_vocab
        self.tag_vocab = tag_vocab

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        src_encoded = self.word_vocab.encode(self.sentences[idx])
        trg_encoded = self.tag_vocab.encode(self.tags[idx])
        
        # Thêm <SOS> và <EOS> cho target (cho Decoder học)
        trg_encoded = [self.tag_vocab.token_to_idx["<SOS>"]] + trg_encoded + [self.tag_vocab.token_to_idx["<EOS>"]]
        
        return torch.tensor(src_encoded), torch.tensor(trg_encoded)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    
    # Pad sequences
    src_padded = pad_sequence(src_batch, padding_value=0, batch_first=True) # 0 is <PAD>
    trg_padded = pad_sequence(trg_batch, padding_value=0, batch_first=True)
    
    return src_padded, trg_padded

In [17]:
# ==========================================
# 4. TRAINING & EVALUATION ENGINE
# ==========================================

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        
        # Bỏ qua token đầu tiên (<SOS>) khi tính loss
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0) # Tắt teacher forcing
            
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def get_f1_score(model, iterator, tag_vocab):
    model.eval()
    pred_tags_all = []
    true_tags_all = []
    
    idx2tag = tag_vocab.idx_to_token
    
    with torch.no_grad():
        for src, trg in iterator:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0)
            predictions = output.argmax(dim=-1)
            
            # Convert indices to tags
            for i in range(predictions.shape[0]):
                p_sent = []
                t_sent = []
                # Bắt đầu từ 1 để bỏ <SOS>
                for j in range(1, trg.shape[1]):
                    token_true_idx = trg[i, j].item()
                    token_pred_idx = predictions[i, j].item()
                    
                    if token_true_idx == tag_vocab.token_to_idx["<EOS>"]:
                        break # Dừng khi gặp EOS của câu gốc
                    
                    # Bỏ qua padding
                    if token_true_idx != tag_vocab.token_to_idx["<PAD>"]:
                        t_sent.append(idx2tag[token_true_idx])
                        p_sent.append(idx2tag[token_pred_idx])
                
                true_tags_all.append(t_sent)
                pred_tags_all.append(p_sent)

    print("\n--- Classification Report ---")
    try:
        print(classification_report(true_tags_all, pred_tags_all))
        f1 = f1_score(true_tags_all, pred_tags_all)
    except Exception as e:
        print(f"Lỗi khi tính F1 (có thể do format tag không đúng chuẩn BIO): {e}")
        f1 = 0
    return f1

# ==========================================
# 5. MAIN EXECUTION
# ==========================================

if __name__ == '__main__':
    # --- Configuration ---
    FILE_PATH = r'../dataset/phonert/syllable/train_syllable.conll'
    BATCH_SIZE = 32
    N_EPOCHS = 10
    HIDDEN_DIM = 256
    N_LAYERS = 1
    EMB_DIM = 100
    DROPOUT = 0.5
    LEARNING_RATE = 0.01
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # --- Load Data ---
    print("Loading data...")
    TRAIN_PATH = r'../dataset/phonert/syllable/train_syllable.conll'
    DEV_PATH   = r'../dataset/phonert/syllable/dev_syllable.conll'
    TEST_PATH  = r'../dataset/phonert/syllable/test_syllable.conll'

    # 1. Load Data (Load riêng lẻ từng tập)
    print("Loading data...")
    train_sents, train_tags = read_phoner_data(TRAIN_PATH)
    val_sents, val_tags     = read_phoner_data(DEV_PATH)
    test_sents, test_tags   = read_phoner_data(TEST_PATH)
    # 2. Build Vocab
    print("Building vocabulary...")
    word_vocab, tag_vocab = build_vocab(train_sents + val_sents, train_tags + val_tags)
    print(f"Word Vocab: {len(word_vocab)}, Tag Vocab: {len(tag_vocab)}")

    # 3. DataLoader
    train_dataset = NERDataset(train_sents, train_tags, word_vocab, tag_vocab)
    val_dataset = NERDataset(val_sents, val_tags, word_vocab, tag_vocab)
    
    train_iterator = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_iterator = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    # 4. Initialize Model
    enc = Encoder(len(word_vocab), EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
    dec = Decoder(len(tag_vocab), EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
    model = Seq2Seq(enc, dec, device).to(device)

    # Init weights
    def init_weights(m):
        for name, param in m.named_parameters():
            nn.init.uniform_(param.data, -0.08, 0.08)
    model.apply(init_weights)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    PAD_IDX = tag_vocab.token_to_idx["<PAD>"]
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    # 5. Training Loop
    best_valid_loss = float('inf')

    print("Start training...")
    for epoch in range(N_EPOCHS):
        start_time = time.time()
        
        train_loss = train(model, train_iterator, optimizer, criterion, clip=1)
        valid_loss = evaluate(model, val_iterator, criterion)
        
        end_time = time.time()
        epoch_mins, epoch_secs = divmod(end_time - start_time, 60)
        
        print(f'Epoch: {epoch+1:02} | Time: {int(epoch_mins)}m {int(epoch_secs)}s')
        print(f'\tTrain Loss: {train_loss:.3f}')
        print(f'\t Val. Loss: {valid_loss:.3f}')
        
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), 'best_model.pt')

    # 6. Evaluation
    print("\nLoading best model to evaluate...")
    model.load_state_dict(torch.load('best_model.pt'))
    f1 = get_f1_score(model, val_iterator, tag_vocab)
    print(f"Final F1-Score: {f1:.4f}")

Using device: cuda
Loading data...
Loading data...
Building vocabulary...
Word Vocab: 4762, Tag Vocab: 25
Start training...
Epoch: 01 | Time: 0m 26s
	Train Loss: 0.949
	 Val. Loss: 1.363
Epoch: 02 | Time: 0m 27s
	Train Loss: 0.832
	 Val. Loss: 1.345
Epoch: 03 | Time: 0m 31s
	Train Loss: 0.804
	 Val. Loss: 1.358
Epoch: 04 | Time: 0m 31s
	Train Loss: 0.796
	 Val. Loss: 1.345
Epoch: 05 | Time: 0m 30s
	Train Loss: 0.792
	 Val. Loss: 1.353
Epoch: 06 | Time: 0m 32s
	Train Loss: 0.795
	 Val. Loss: 1.359
Epoch: 07 | Time: 0m 38s
	Train Loss: 0.797
	 Val. Loss: 1.373
Epoch: 08 | Time: 0m 40s
	Train Loss: 0.784
	 Val. Loss: 1.388
Epoch: 09 | Time: 0m 37s
	Train Loss: 0.788
	 Val. Loss: 1.388
Epoch: 10 | Time: 0m 23s
	Train Loss: 0.790
	 Val. Loss: 1.364

Loading best model to evaluate...

--- Classification Report ---
                     precision    recall  f1-score   support

                AGE       0.00      0.00      0.00       361
               DATE       0.00      0.00      0.00      1

In [16]:
# --- DEBUG BLOCK ---
def print_debug_predictions(model, iterator, word_vocab, tag_vocab, limit=3):
    model.eval()
    idx2word = word_vocab.idx_to_token
    idx2tag = tag_vocab.idx_to_token
    
    count = 0
    with torch.no_grad():
        for src, trg in iterator:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0) # Tắt teacher forcing
            predictions = output.argmax(1) # [batch, len]
            
            # In ra vài câu đầu tiên của batch
            for i in range(src.shape[0]):
                if count >= limit: return
                
                print(f"\n--- Câu thứ {count + 1} ---")
                
                # Lấy lại câu gốc (bỏ padding)
                sent = [idx2word[idx.item()] for idx in src[i] if idx.item() != 0]
                
                # Lấy nhãn thật
                real_tags = [idx2tag[idx.item()] for idx in trg[i] if idx.item() not in [0, 1, 2]] # Bỏ PAD, SOS, EOS
                
                # Lấy nhãn dự đoán (cắt cho bằng độ dài thật)
                # Lưu ý: prediction có thể dài hơn hoặc ngắn hơn do logic Decoder, 
                # nhưng ở đây ta đang dùng seq_len cố định theo batch nên OK.
                pred_tags_raw = predictions[i][1:] # Bỏ SOS đầu tiên
                pred_tags = []
                for idx in pred_tags_raw:
                    if len(pred_tags) < len(real_tags): # Chỉ lấy đúng độ dài câu
                        tag_str = idx2tag.get(idx.item(), "<UNK>")
                        pred_tags.append(tag_str)
                
                print(f"Sentence: {' '.join(sent)}")
                print(f"True Tags: {real_tags}")
                print(f"Pred Tags: {pred_tags}")
                
                count += 1

print("Đang kiểm tra model dự đoán cái gì...")
print_debug_predictions(model, val_iterator, word_vocab, tag_vocab)

Đang kiểm tra model dự đoán cái gì...

--- Câu thứ 1 ---
Sentence: Bác sĩ Nguyễn Trung Nguyên , Giám đốc Trung tâm Chống độc , Bệnh viện Bạch Mai , cho biết bệnh nhân được chuyển đến bệnh viện ngày 7/3 , chẩn đoán ngộ độc thuốc điều trị sốt rét chloroquine .
True Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DATE', 'O', 'O', 'O', 'B-SYMPTOM_AND_DISEASE', 'I-SYMPTOM_AND_DISEASE', 'I-SYMPTOM_AND_DISEASE', 'O', 'O', 'O', 'O', 'O', 'O']
Pred Tags: ['<PAD>', '<UNK>', '<PAD>', '<SOS>', '<SOS>', '<EOS>', '<PAD>', '<PAD>', '<SOS>', 'B-GENDER', '<EOS>', '<UNK>', 'B-LOCATION', '<SOS>', '<UNK>', '<PAD>', '<PAD>', '<PAD>', 'B-SYMPTOM_AND_DISEASE', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']

--- Câu thứ 2 ---
Sentence: " Bệnh nhân 812 " , nam , 62 tuổi , là nhân viên giao bánh ti